# PPE 탐지 7차-A 학습 — 현장 안전모 실명 해소 (yolo26 · 8클래스)

**v5 를 기준선으로 GDUT + v5 person·vest 의사라벨을 더한다.** 507 은 뺀다(6차 퇴보의 원인 — 흰 안전모 착용 테이크가 전부 무라벨).

| 근거 | 값 |
|---|---|
| v5 현장 35장 helmet(중심포함) | **0/100** — 예측 자체가 0개 |
| pilot3 (GDUT+의사라벨 · 5ep) | **94/100 · FP 2** · person 96/113 · vest 69/100 |
| GDUT 단독 (의사라벨 없음) | helmet .614 얻고 **person .973→.000** — 라벨 없는 클래스 = 배경 |

⟹ 처방은 검증됐다. 이 노트북은 그 처방을 30 에포크로 완주한다.

**v6 에서 바뀐 것 4가지** (그 밖은 그대로 — 하이퍼를 건드리면 자료 효과와 조건 효과가 안 갈린다)
1. `USE_A507 = False` · 9번째 클래스 `no_safety_shoes` 제거 → **8클래스**
2. GDUT 출처 추가 — E63 과 같은 tar 방식 · fraction 1.0 · 매핑 불필요(이미 통합 8클래스 id)
3. person 의사라벨 셀이 `gdut_` 파일에 person 을 **또** 붙이지 않는다(이미 들어 있다)
4. 학습 뒤 라벨 수를 `train_v7.py` 의 expect 범위로 검증한다

**게이트는 mAP 가 아니다.** 6차가 mAP 오르며 오경보 +53% 였다. 판정 정본 = `train_v7.py` `GATE_COUNTS`(박스 개수). 현장 35장·ppe.sample 은 **시험 전용이라 Drive 에 없다** — 채점은 [CCTV/ML] 이 정본 자로 독립 수행한다.


## 0. 어느 계정에서 돌리는가

**★ 09-14 정정 — 원 계정(changwoojin)에 Colab Pro 가 붙어 있고 컴퓨팅 단위가 100개 있다.**
A100 도 고를 수 있다. 종전의 「단위 소진 → 팀원 계정」 전제는 **더 이상 맞지 않는다**
(09-13 에 A100 이 회색이었던 것은 그 시점 상태였다).

⟹ **원 계정에서 직접 돌린다.** 아래 표는 팀원 계정으로 넘길 때만 필요하다.

⚠ **단위 100개는 A100 한 번 분량이다.** A100 은 시간당 대략 8~15단위를 쓰므로(정확한
값은 화면에서 확인) 4.8시간 학습에 약 40~70단위다. **재실행 여유가 많지 않다** —
설정·라벨 수 검산까지 통과한 뒤에 학습을 시작할 것.

| 순서 | 누가 | 할 것 |
|---|---|---|
| 1 | 원 계정 | Drive `vitalguard-ppe` 폴더 우클릭 → 공유 → 팀원 계정을 **편집자**로 추가 |
| 2 | 원 계정 | 이 노트북(`.ipynb`)을 `vitalguard-ppe/` 안에 업로드 → 팀원이 Colab 에서 바로 연다 |
| 3 | 팀원 | Drive `공유 문서함` → `vitalguard-ppe` 우클릭 → **정리 → 바로가기 추가** → `내 드라이브` **루트** |
| 4 | 팀원 | Kaggle API 토큰 **본인 것** 발급(`kaggle.com/settings` → API) + [SH17 페이지](https://www.kaggle.com/datasets/mugheesahmad/sh17-dataset-for-ppe-detection)에서 Download 를 1회 눌러 규정 동의 |
| 5 | 팀원 | Roboflow API 키 **본인 것** 발급 · 설정 셀 `RUNNER` 에 본인 이름 |

3번이 핵심이다. 바로가기가 내 드라이브 루트에 있으면 `/content/drive/MyDrive/vitalguard-ppe`
경로가 그대로 동작한다. 설정 셀이 `e63_yolo.tar`(2.06GB) 존재를 검사하므로 빠졌으면 그 자리에서 드러난다.

- Kaggle·Roboflow 자격증명은 계정 귀속이라 **공유하지 말고 각자 것**을 쓴다
- 산출물(체크포인트·가중치·평가 결과)은 공유 폴더에 쌓여 **양쪽 계정에서 보인다**

### ★★ 학습과 평가를 같은 세션에서 끝낼 것

평가를 나중에 따로 하려면 `/content` 가 비워져 있어 **데이터셋을 30분 재구축**해야 한다
(4차에서 실제로 겪었다). 재구축은 컴퓨팅 단위를 또 쓴다 — 단위 100개에서는 아까운 비용이다.

**학습(5장)이 끝나면 런타임을 끊지 말고 6~10장을 이어서 돌린다.** 10~20분이면 끝난다.
평가표는 `vitalguard-ppe/results/<RUN_NAME>/` 에 **CSV 로 자동 저장**되므로
화면 캡처에 의존하지 않는다 — 세션이 죽어도 숫자는 남고 원 계정에서 그대로 읽는다.

### ⛔ 시작 전 확인 — 잔여 컴퓨팅 단위

**★ 기준은 A100 batch 64 다.** v5·v6 가 그 조건이라, 64 로 돌아야 채점 결과를 v5·v6 와
바로 비교할 수 있다. `BATCH = None` 이면 설정 셀의 자동 감지가 A100 에서 64 를 고른다.

**★ 09-14 — 원 계정에서 A100 을 고를 수 있다**(Colab Pro · 단위 100개 확인).
09-13 에 회색이었던 것은 그 시점 상태였고, 지금 기준은 그때 관찰이 아니라 **화면의 현재 값**이다.

⚠ 다만 **유형을 A100 으로 골라도 혼잡하면 T4 가 붙는다.** 09-13 CLI 에서는 배정된 A100 이
70분 만에 회수되기까지 했다. 그래서 **`nvidia-smi` 출력과 설정 셀의 기준 판정 줄**을
둘 다 눈으로 확인한다 — 「골랐다」와 「붙었다」는 다른 층이다.

T4 가 붙으면 자동 감지가 batch 32 를 고른다. 실효 배치는 accumulate 2 로 64 가 유지되지만
**BN 통계는 보정되지 않는다** — 기준 조건이 아니므로 채점 보고에 각주를 달 것.
`EPOCHS` 축소는 비권장 — 3차 20에포크가 부족하다는 것이 이미 확인됐다.

### 동시 실행은 금지다

학습 셀은 `project=` 를 공유 Drive 로 쓰고 `exist_ok=True` 다. 두 계정이 같은 `RUN_NAME` 을
동시에 돌리면 `last.pt`·`best.pt`·`results.csv` 를 서로 덮어쓴다. **오류가 나지 않으므로**
몇 시간 뒤에 조용히 망가진 것을 발견한다. `RUNNER` 를 적어두면 **학습 직전 잠금 셀**이
상대 세션이 살아있는지(`last.pt` 갱신 시각으로) 판정해 막는다.

절차 전체는 [RUNBOOK-5차-협업.md](RUNBOOK-5차-협업.md) 참조.


## 1. 런타임 확인

`런타임 → 런타임 유형 변경 → **A100 GPU**`. **맨 위부터 순서대로 실행할 것.**

★ **A100 이 기준이다**(v5·v6 와 같은 조건). 아래 `nvidia-smi` 출력에서 **`A100` 글자를
눈으로 확인할 것** — 런타임 유형을 A100 으로 골라도 혼잡하면 T4 가 붙을 수 있다.
`Tesla T4` 가 보이면 A100 이 아니다. 그대로 진행해도 학습은 되지만 기준 조건이 아니다.

In [ ]:
!nvidia-smi
!free -g | head -2
import sys; print('python', sys.version.split()[0])

In [ ]:
%pip -q install "ultralytics>=8.4.0" kaggle roboflow
import ultralytics; ultralytics.checks()

## 2. 설정

### 예상 소요

| 구성 | GPU · batch | train | 에포크당 | 30에포크 | 출처 |
|---|---|---|---|---|---|
| **★기준 — 7차-A 계획** | **A100 · 64** | 18,816 | 약 9.5분 | **약 4.8시간** | 4차 실측 환산 |
| 대체 — T4 | T4 · 32 | 18,816 | 미측정 | 미측정 | — |
| (참고) 09-13 중단분 | T4 · 16 | 18,816 | **20.4분** | 약 10.2시간 | **09-13 실측** |
| (참고) 5차 | A100 · 64 | 약 18,700 | 약 9.5분 | 약 4.8시간 | 4차 실측 환산 |
| (참고) 4차 | A100 · 64 | 약 17,200 | 8.7분 | 4.4시간 | 4차 실측 |

★ **기준은 A100 batch 64** — v5·v6 가 그 조건이라 결과를 그대로 비교할 수 있다.
⚠ 유형을 A100 으로 골라도 **혼잡하면 T4 가 붙는다.** 실제로 붙은 GPU 를 `nvidia-smi` 로 확인할 것.

★ **굵은 줄만 실측이다.** T4 20.4분은 `results.csv` 의 `time` 열에서 읽은 벽시계값이고,
에포크 1 이라 데이터셋 스캔·캐시 같은 1회성 비용이 섞여 있다(= 보수적).
화면의 `it/s` 로 환산하지 말 것 — 검증 시간이 빠져 있어 09-13 에 두 번 틀렸다.

★ batch 32 의 에포크 시간은 **아직 아무도 재지 않았다.** 16 보다 빠를 수도 느릴 수도 있다.
첫 에포크가 끝나면 위 표의 「미측정」을 실제 값으로 채울 것.

SH17 `head` 선별 채택·`person` 자동 라벨은 **이미지 수를 늘리지 않는다**(라벨만 늘어난다)
→ 에포크 시간은 위 표와 같다. 자동 라벨 생성은 1회 3~6분(A100).


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# ---- 설정 -------------------------------------------------------------
MODEL   = 'yolo26s.pt'
IMGSZ   = 640          # 1·3차 검증됨. 960 은 에포크당 31분이라 완주 불가였다
EPOCHS  = 30           # 3차 20에포크는 손실이 끝까지 하락 중이라 부족했다
BATCH   = None         # ★ 09-14 — **기준은 A100 batch 64** 다. v5·v6 가 그 조건이라, 64 로 돌아야
                       #   채점 결과를 v5·v6 와 바로 비교할 수 있다. None 이면 아래 자동 감지가 고른다.
                       #
                       #     A100 → 64   ← ★기준. 이것으로 돌리는 것이 계획이다
                       #     T4   → 32   ← 대체. 64 는 T4 에 안 들어간다
                       #
                       #   ⚠ T4 대체값의 근거 — 09-13 T4 실측이 batch 16 에서 6.85G 였다. 활성값이
                       #     배치에 비례하므로 32 는 약 13G 로 T4 15GB 의 대부분을 쓴다. 64 는 약 27G 라 OOM.
                       #     32 면 accumulate 2 로 실효 배치 64 가 유지돼(옵티마이저·weight_decay·lr 일정 동일)
                       #     **BN 통계만** 기준과 달라진다. OOM 이면 첫 1~2분 안에 나며 그때만 24 로 낮춘다.
                       #   ⚠ T4 로 돌린 결과는 기준 조건이 아니다 — 채점 보고에 각주를 달 것.
                       #   ⚠ A100 에서 돌린다면 64 로 되돌려 v5·v6 와 완전히 같게 맞출 것.
WORKERS = None         # None = GPU 자동 감지. 데이터 공급 병목을 좌우한다
RESUME  = False        # 세션 끊긴 뒤 이어서 학습할 때만 True (다른 값은 건드리지 말 것)
EVAL_ONLY = False      # ★ 09-14 신설. 학습은 끝났고 **평가만** 다시 돌릴 때 True.
                       #   True 면 학습 셀이 best.pt 경로만 잡고 학습을 건너뛴다.
                       #   ⚠ 이게 없으면 학습 셀을 실수로 눌렀을 때 RESUME=False 라
                       #     0에포크부터 다시 학습하며 **Drive 의 best.pt 를 덮어쓴다.**
RUNNER  = ''           # ★ 협업 필수 — 본인 짧은 이름(예 'changwoo', 'teammate').
                       #   학습 직전 잠금 셀이 이걸로 동시 실행을 막는다
SCALE   = 0.8          # ★ 5차 신규: 학습 배율 증강(ultralytics 기본 0.5). 근거리 머리(높이
                       #   0.19~0.31)가 학습 분포(중앙값 0.051·최대 0.141) 밖이라 못 잡던
                       #   것의 대응. 설계 밴드 0.7~0.9 의 중간값

# 출처별 비율 (위 표 참조)
SH17_FRACTION = 0.5    # 3차 0.3 에서 foot 499개로 학습 불가였다.
                       # head·helmet 매핑을 뺐으므로 올려도 균형을 해치지 않는다
CSS_FRACTION  = 1.0    # safety_vest 유일 출처. 규모가 작아 전량
HHW_FRACTION  = 0.45   # ★ 5차: 0.15→0.45. 4차 감축이 helmet R 0.821→0.433 을 만들었다
                       #   (HHW 가 3차 helmet 의 69% 공급원). 3차 0.5 는 편중이 심했으므로 직전까지만
E63_FRACTION  = 0.4    # ★ 배포 도메인. 라벨이 배타적이라 비중을 올린다
GDUT_FRACTION = 1.0    # ★ 7차-A 신규. 2,531장 전량. val 643 은 tar 안에서 해시 홀드아웃(GT 만)

# ★ 홀드아웃 예약 — 학습 전에 시퀀스를 통째로 뗀다.
#   3차는 프레임 단위로 나눠 43개 시퀀스 전부가 학습에 노출돼 채점이 불가능했다.
E63_HOLDOUT_SEQ_RATIO = 0.30   # 시퀀스 43개 중 12개. 0.18 이면 홀드아웃이 420장뿐이라 부족했다

# ★★ 5차 핵심 — 도메인 얽힘 해소 (07-30 GoPro 검증에서 확정된 4차의 진짜 결함)
SH17_HEAD_FILTER  = True   # SH17 head 중 helmet 과 겹치지 않는 것만 채택(= 미착용 머리)
SH17_HEAD_IOU     = 0.3    # 겹침 판정 임계. 이보다 크게 겹치면 '안전모 쓴 머리'로 보고 버린다
E63_PERSON_PSEUDO = True   # E63·HHW 에 person 자동 라벨(COCO 사전학습) — train 분할만
PSEUDO_CONF       = 0.5    # 자동 라벨 신뢰도 하한. 낮추면 오라벨이 늘어난다

USE_SH17 = True        # ⚠️ CC BY-NC-SA(비상업)
USE_CSS  = True        # CC BY 4.0
USE_HHW  = True
USE_E63  = True        # ⚠️ AI Hub — 학습용 한정(상업 이용은 수행기관 별도 협의)
USE_GDUT = True        # ★ 7차-A — GDUT GT(helmet·head) + v5 의사라벨(person·vest)
USE_A507 = False       # ⛔ 7차 — 6차 퇴보 원인(흰 안전모 무라벨). 라벨 수 검증이 부재를 확인한다
# -----------------------------------------------------------------------

NC = USE_SH17 or USE_E63                    # 비상업 데이터 포함 여부 (GDUT 라이선스는 별도 확인 — 어차피 nc)
RUN_NAME  = f"{Path(MODEL).stem}_{IMGSZ}_v7a" + ("_nc" if NC else "")   # ★ run A. run B 는 v7b
def _resolve_drive():
    """작업 폴더를 이름이 아니라 "e63_yolo.tar 를 가진 폴더" 로 찾는다.

    내 드라이브에 같은 이름이 이미 있으면 Drive 는 공유 폴더 바로가기에 " (1)" 을 붙인다.
    팀원 계정에서 실제로 이 일이 일어나 경로가 조용히 엉뚱한 폴더를 가리켰다
    (옛 vitalguard-ppe 에는 runs 만 있었다). 이름 하드코딩으로는 잡히지 않는다.
    """
    md = Path('/content/drive/MyDrive')
    canon = md / 'vitalguard-ppe'
    if not md.is_dir():
        return canon
    cands = sorted(p for p in md.iterdir()
                   if p.is_dir() and p.name.startswith('vitalguard-ppe'))
    with_tar = [p for p in cands if (p / 'datasets' / 'e63_yolo.tar').exists()]
    if with_tar:
        if with_tar[0] != canon:
            print(f'★ 경로 자동 해결 — {[p.name for p in cands]} 중 '
                  f'"{with_tar[0].name}" 선택 (e63_yolo.tar 보유)')
        return with_tar[0]
    return canon


DRIVE     = _resolve_drive()
RUNS      = DRIVE / 'runs'
E63_TAR   = DRIVE / 'datasets' / 'e63_yolo.tar'   # 맥에서 변환해 올린 단일 아카이브
E63_SRC   = Path('/content/e63_yolo')             # 여기로 풀어서 쓴다(로컬 디스크)
GDUT_TAR  = DRIVE / 'datasets' / 'gdut_yolo.tar'  # ★ 7차-A · 713MB · [CCTV/ML] 09-13 스테이징 · sha256 동봉
GDUT_SRC  = Path('/content/gdut_yolo')
RAW       = Path('/content/raw')
DATA      = Path('/content/ppe3')
DATA_YAML = DATA / 'data.yaml'
for p in (DRIVE, RUNS, RAW, DATA):
    p.mkdir(parents=True, exist_ok=True)

# ---- GPU 자동 튜닝 -----------------------------------------------------
# 1차(T4)에서 GPU 가 71% 놀았다. 원인은 T4 인스턴스의 CPU 2코어가 이미지 공급을
# 못 따라간 것. A100/L4 인스턴스는 CPU 코어가 많아 workers 를 올려야 성능이 난다.
import os, torch
_gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
_cpu = os.cpu_count() or 2
if BATCH is None or WORKERS is None:
    if 'A100' in _gpu:      _b, _w = 64, min(8, _cpu)
    elif 'H100' in _gpu:    _b, _w = 96, min(8, _cpu)
    elif 'L4' in _gpu:      _b, _w = 32, min(6, _cpu)
    elif 'V100' in _gpu:    _b, _w = 32, min(4, _cpu)
    else:                   _b, _w = 32, min(2, _cpu)      # T4 등 — ★09-14 16→32 (BN 격차 절반)
    BATCH   = _b if BATCH   is None else BATCH
    WORKERS = _w if WORKERS is None else WORKERS
print(f'GPU  : {_gpu} · CPU {_cpu}코어 → batch {BATCH} · workers {WORKERS}')
# ★ 09-14 — 기준 조건(A100 batch 64)으로 돌고 있는지 여기서 소리내어 말한다.
#   09-13 에는 batch 가 조용히 16 으로 정해졌고, 학습이 끝날 때까지 아무도 몰랐다.
if BATCH == 64:
    print('       ✅ 기준 조건 — v5·v6(A100 batch 64)과 같다. 결과를 그대로 비교할 수 있다')
else:
    print(f'       ⚠️ 기준(batch 64) 이 아니다 → batch {BATCH}. 실효 배치는 accumulate 로 보정되지만')
    print( '          BN 통계는 보정되지 않는다. 채점 보고에 각주를 달 것')
if 'A100' in _gpu or 'H100' in _gpu:
    print('       ★ 컴퓨팅 단위 소모가 T4 의 약 6배다. 남은 단위를 확인할 것')
# -----------------------------------------------------------------------

print('run  :', RUN_NAME)
print('E63  :', E63_TAR, '→',
      f'{E63_TAR.stat().st_size/1e9:.2f} GB' if E63_TAR.exists()
      else '⚠️ 없음 — Drive 동기화, 팀원 계정이면 0장의 공유·바로가기부터 확인')
print('GDUT :', GDUT_TAR, '→',
      f'{GDUT_TAR.stat().st_size/1e6:.0f} MB' if GDUT_TAR.exists()
      else '⚠️ 없음 — Drive vitalguard-ppe/datasets/gdut_yolo.tar 동기화 확인')
# ★ runs 쓰기 권한 확인 — 공유 폴더가 '뷰어' 면 4.8시간 학습을 다 하고 나서
#   체크포인트 저장에서 터진다. 여기서 잡으면 몇 초로 끝난다.
try:
    _probe = RUNS / '.write_test'
    _probe.write_text('ok')
    _probe.unlink()
    print('runs :', RUNS, '(쓰기 가능)')
except OSError as _e:
    print(f'\n⚠️ runs 쓰기 불가 ({_e.__class__.__name__}) — 공유 권한이 뷰어일 가능성.')
    print('   편집자로 바꾸지 않으면 학습이 끝나는 순간 체크포인트를 잃는다.')

if NC:
    print('\n⚠️  비상업 데이터 포함 — 산출 가중치는 상용 빌드에 반입하지 말 것(-nc 접미사 유지).')

## 3. 데이터셋 다운로드

- **SH17** — Kaggle. **API 토큰 문자열** 필요(`kaggle.com/settings` → API → Create New Token, **한 번만** 표시)
- **CSS · HHW** — Roboflow. API 키 필요
- **E63** — 다운로드 없음. Drive 에 이미 YOLO 형식으로 있다

In [ ]:
def _ensure(pkg, mod=None):
    """패키지가 없으면 설치한다 — 설치 셀을 건너뛰어도 동작하도록.

    ★ import 하지 않고 존재만 확인한다. kaggle 은 import 시점에 인증을 시도해
      자격증명이 없으면 그 자리에서 죽는다(exit(1) → NameError).
    """
    import importlib, importlib.util, subprocess, sys
    importlib.invalidate_caches()
    if importlib.util.find_spec(mod or pkg) is None:
        print(f'{pkg} 설치 중...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
        importlib.invalidate_caches()


if USE_SH17:
    import os
    from getpass import getpass

    # 자격증명을 먼저 만든 뒤 kaggle 을 부른다 — 순서가 뒤바뀌면 인증 오류로 죽는다
    tok = getpass('Kaggle API token: ').strip()
    os.environ['KAGGLE_API_TOKEN'] = tok
    kdir = Path.home() / '.kaggle'
    kdir.mkdir(exist_ok=True)
    (kdir / 'access_token').write_text(tok)
    (kdir / 'access_token').chmod(0o600)
    del tok

    _ensure('kaggle')
    !kaggle datasets download -d mugheesahmad/sh17-dataset-for-ppe-detection -p /content/raw --unzip -q
    print('SH17 다운로드 완료')

# 403 이면 브라우저에서 데이터셋 페이지를 열고 Download 를 눌러 규정에 동의할 것

In [ ]:
_ensure('roboflow')
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass('Roboflow API key: '))

if USE_CSS:
    css = (rf.workspace('roboflow-universe-projects')
             .project('construction-site-safety')
             .version(27).download('yolov8', location='/content/raw/css'))
    print('CSS →', css.location)

if USE_HHW:
    # v13 이 helmet/head 분리 버전(v14 는 한 클래스로 병합돼 있어 못 씀)
    hhw = (rf.workspace('joseph-nelson')
             .project('hard-hat-workers')
             .version(13).download('yolov8', location='/content/raw/hhw'))
    print('HHW →', hhw.location)

In [ ]:
# ★ 실제 구조와 클래스 순서를 눈으로 확인한다. 배포본마다 다르므로 반드시 대조할 것.
!find /content/raw -maxdepth 3 -type d | head -20
print()
for y in sorted(Path('/content/raw').rglob('*.yaml')):
    print('---', y)
    print(y.read_text()[:260])
print('--- E63 (Drive, 변환 완료본) ---')
print((E63_SRC / 'data.yaml').read_text() if (E63_SRC / 'data.yaml').exists() else '없음')

## 4. 통합 8클래스로 리매핑

```
0 person   1 head   2 helmet   3 safety_vest
4 foot     5 shoes  6 hands    7 gloves
```

미착용은 클래스로 두지 않고 `head`(맨머리) + person 포함관계로 코드에서 판정한다.

**처리 세 가지**

- **증강 사본 제거** — Roboflow 는 `<원본>_jpg.rf.<해시>` 로 증강본을 만든다. 원본 1장만 남긴다
- **출처별 서브샘플** — 파일명 해시로 결정적으로 고른다(재실행·재개해도 동일)
- **E63 는 이미 YOLO** — 변환 없이 비율만 적용해 연결한다

In [ ]:
import hashlib, re, shutil, yaml
from collections import Counter
from pathlib import Path

UNIFIED = ['person', 'head', 'helmet', 'safety_vest', 'foot', 'shoes',
           'hands', 'gloves']   # ★ 7차 — 8클래스로 복귀(no_safety_shoes 보류 · 엣지 NAMES 와 일치)
UNIFIED_IDX = {n: i for i, n in enumerate(UNIFIED)}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def _norm(s):
    return ''.join(ch for ch in str(s).lower() if ch.isalnum())


# ★ 5차 — SH17 의 head 를 되살린다. 단 '겹치지 않는 것만'.
#   SH17 은 머리에 head 를, 안전모에 helmet 을 따로 붙여 둘이 78.2% 겹친다
#   (E63·CSS·HHW 는 '미착용=head / 착용=helmet' 로 배타적). 4차는 이 충돌을 피해
#   SH17 의 head·helmet 을 통째로 뺐는데, 그 결과 근거리·일반 환경의 머리를
#   학습하지 못해 E63 도메인 밖에서 head 가 0건이 됐다(07-30 GoPro 검증).
#   겹치지 않는 head 는 그 자체로 '안전모 안 쓴 머리' 이므로 우리 정의와 일치한다.
#   → 아래 sh17_head_filter 가 remap 시점에 골라낸다.
if SH17_HEAD_FILTER:
    SH17_MAP = {'person': 'person', 'head': 'head', 'helmet': 'helmet',
                'safetyvest': 'safety_vest', 'foot': 'foot', 'shoes': 'shoes',
                'hands': 'hands', 'gloves': 'gloves'}
else:   # 4차와 동일 — SH17 은 head·helmet 을 기여하지 않는다
    SH17_MAP = {'person': 'person', 'safetyvest': 'safety_vest', 'foot': 'foot',
                'shoes': 'shoes', 'hands': 'hands', 'gloves': 'gloves'}

# 출처마다 라벨링한 클래스가 다르다. 라벨이 없는 출처의 이미지에서는 정탐도 오탐으로
# 계산되므로(3차에서 person P 0.493 로 찍혔으나 실제는 0.891) 평가 때 대상을 제한한다.
SOURCE_CLASSES = {
    'sh17': ({'person', 'foot', 'shoes', 'hands', 'gloves', 'safety_vest', 'head'}
             if SH17_HEAD_FILTER else
             {'person', 'foot', 'shoes', 'hands', 'gloves', 'safety_vest'}),
    'css':  {'person', 'head', 'helmet', 'safety_vest'},
    'hhw':  {'head', 'helmet'},
    'e63':  {'head', 'helmet'},
    # ★ 7차-A — GDUT. GT 는 helmet·head, person·safety_vest 는 v5 의사라벨(train 만).
    #   val 은 GT 만이라 fair_eval 에서 person·vest 를 gdut 로 재면 정탐이 오탐이 된다 → 둘을 뺀다.
    #   shoes·hands·gloves 는 의사라벨도 없다(v5 shoes P .484) — 이 출처로 재지 않는다.
    'gdut': {'helmet', 'head'},
}
CSS_MAP = {'person': 'person', 'hardhat': 'helmet', 'nohardhat': 'head',
           'safetyvest': 'safety_vest'}
HHW_MAP = {'helmet': 'helmet', 'head': 'head', 'person': 'person'}

# ★ SH17 공식 클래스 순서 — sh17.yaml 원문. 논문/README 나열 순서와 다르다.
SH17_NAMES = ['person', 'ear', 'ear-mufs', 'face', 'face-guard', 'face-mask', 'foot',
              'tool', 'glasses', 'gloves', 'helmet', 'hands', 'head', 'medical-suit',
              'shoes', 'safety-suit', 'safety-vest']

RF_AUG = re.compile(r'^(.*?)_(?:jpg|jpeg|png)\.rf\.[0-9a-f]+$', re.I)


def rf_base(stem):
    """증강 사본들의 공통 원본 이름. 증강본이 아니면 그대로 반환."""
    m = RF_AUG.match(stem)
    return m.group(1) if m else stem


def load_names(data_yaml):
    d = yaml.safe_load(Path(data_yaml).read_text())['names']
    return [d[k] for k in sorted(d)] if isinstance(d, dict) else list(d)


def find_yaml(root, exclude=(), prefer=('sh17.yaml', 'data.yaml')):
    cands = [p for p in Path(root).rglob('*.yaml') if not any(e in p.parts for e in exclude)]
    for name in prefer:
        for p in cands:
            if p.name == name:
                return p
    return cands[0] if cands else None


def build_index_map(src_names, name_map):
    out = {}
    for i, n in enumerate(src_names):
        tgt = name_map.get(_norm(n))
        out[i] = UNIFIED_IDX[tgt] if tgt else None
    return out


def _hash1000(s):
    return int(hashlib.md5(s.encode()).hexdigest()[:8], 16) % 1000


def deterministic_split(stem, val_ratio=0.1):
    return 'val' if _hash1000(stem) < val_ratio * 1000 else 'train'


def _place(img, lines, dst_root, sp, stem, link=True):
    """이미지 링크 + 라벨 파일 기록."""
    io_ = Path(dst_root) / 'images' / sp
    lo_ = Path(dst_root) / 'labels' / sp
    io_.mkdir(parents=True, exist_ok=True)
    lo_.mkdir(parents=True, exist_ok=True)
    di = io_ / (stem + img.suffix)
    if not di.exists():
        if link:
            try:
                di.symlink_to(img.resolve())
            except OSError:
                shutil.copy2(img, di)
        else:
            shutil.copy2(img, di)
    (lo_ / (stem + '.txt')).write_text('\n'.join(lines) + '\n')


def collect_pairs(root, val_ratio=0.1, dedupe_aug=True, fraction=1.0):
    """(이미지, 라벨, split) 목록. 증강 사본 제거·서브샘플 적용."""
    root = Path(root)
    seen, pairs, n_aug, n_frac = set(), [], 0, 0
    for img in sorted(root.rglob('*')):
        if img.suffix.lower() not in IMG_EXT or not img.is_file() or img.name.startswith('.'):
            continue   # 숨김·AppleDouble(._*) 제외
        pl = [p.lower() for p in img.parts]
        if 'images' not in pl:
            continue
        base = rf_base(img.stem)
        if dedupe_aug:
            if base in seen:
                n_aug += 1
                continue
            seen.add(base)
        if fraction < 1.0 and _hash1000('frac' + base) >= fraction * 1000:
            n_frac += 1
            continue
        i = len(pl) - 1 - pl[::-1].index('images')
        lbl = Path(*img.parts[:i], 'labels', *img.parts[i + 1:]).with_suffix('.txt')
        s = set(pl)
        sp = ('train' if 'train' in s else
              'val' if s & {'val', 'valid'} else
              'test' if 'test' in s else
              deterministic_split(base, val_ratio))
        pairs.append((img, lbl, sp))
    return pairs, n_aug, n_frac


def _iou_xywhn(a, b):
    """정규화 (cx, cy, w, h) 두 박스의 IoU."""
    ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2
    ax2, ay2 = a[0] + a[2] / 2, a[1] + a[3] / 2
    bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2
    bx2, by2 = b[0] + b[2] / 2, b[1] + b[3] / 2
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0


def sh17_head_filter(items, thresh):
    """★ SH17 전용 — helmet 과 겹치지 않는 head 만 남긴다.

    SH17 은 같은 머리에 head(착용 무관 머리)와 helmet(안전모 물체)을 동시에 붙여
    다른 출처(E63·CSS·HHW = 미착용/착용 배타적)와 의미가 충돌한다(helmet 중 78.2%
    가 head 와 겹침). 겹치지 않는 head 는 '안전모 안 쓴 머리' 이므로 우리 정의와
    일치한다. 겹치는 head 와 helmet 박스 자체는 버린다(박스 정의가 달라 못 쓴다).
    """
    H, M = UNIFIED_IDX['head'], UNIFIED_IDX['helmet']
    helmets = [b for i, b in items if i == M]
    out, drop_head, drop_helmet = [], 0, 0
    for i, b in items:
        if i == M:
            drop_helmet += 1
            continue
        if i == H and any(_iou_xywhn(b, hb) > thresh for hb in helmets):
            drop_head += 1
            continue
        out.append((i, b))
    return out, drop_head, drop_helmet


def remap(pairs, dst_root, idx_map, prefix, link=True, head_filter=False):
    """원본 라벨을 통합 인덱스로 다시 쓰고 이미지를 모은다."""
    kept = dropped = 0
    counts = Counter()
    filt = Counter()
    for img, lbl, sp in pairs:
        items = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                try:
                    new = idx_map.get(int(float(parts[0])))
                    box = tuple(float(x) for x in parts[1:5])
                except ValueError:
                    continue
                if new is None:
                    continue
                items.append((new, box))
        if head_filter:
            items, dh, dm = sh17_head_filter(items, SH17_HEAD_IOU)
            filt['head'] += dh
            filt['helmet'] += dm
        lines = []
        for i, b in items:
            lines.append(f"{i} " + ' '.join(f'{v:.6f}' for v in b))
            counts[UNIFIED[i]] += 1
        if not lines:
            dropped += 1
            continue
        _place(img, lines, dst_root, sp, f'{prefix}_{img.stem}', link)
        kept += 1
    if filt:
        print(f"   ★ SH17 필터 — 안전모 쓴 머리 제외 {filt['head']:,} · "
              f"helmet 박스 제외 {filt['helmet']:,}")
    return kept, dropped, counts


def seq_of(stem):
    """E63 프레임 이름에서 촬영 시퀀스 키를 뽑는다.

    e63_E63_DATA3_H1_L5_D2023-10-26-11-55_082_001290 → ..._082
    같은 시퀀스의 인접 프레임은 거의 동일하므로 분할은 반드시 시퀀스 단위로 한다.
    """
    return stem.rsplit('_', 1)[0]


def e63_reserve_seqs(src, ratio):
    """홀드아웃으로 예약할 시퀀스 집합을 결정적으로 고른다."""
    seqs = set()
    for sp in ('train', 'val'):
        d = Path(src) / 'images' / sp
        if not d.is_dir():
            continue
        for p in d.iterdir():
            if p.suffix.lower() in IMG_EXT and not p.name.startswith('.'):
                seqs.add(seq_of(p.stem))
    return seqs, {s for s in seqs if _hash1000('seq' + s) < ratio * 1000}


def link_yolo(src, dst_root, fraction=1.0, link=True, exclude_seqs=frozenset()):
    """이미 통합 8클래스 YOLO 형식인 데이터셋을 비율만 적용해 연결한다(E63 용).

    라벨 인덱스가 이미 통합 기준이므로 재매핑하지 않는다.
    exclude_seqs 에 든 시퀀스는 통째로 건너뛴다(홀드아웃 예약분).
    """
    src = Path(src)
    kept = skipped = held = 0
    counts = Counter()
    for sp in ('train', 'val', 'test'):
        idir = src / 'images' / sp
        if not idir.is_dir():
            continue
        for img in sorted(idir.iterdir()):
            if img.suffix.lower() not in IMG_EXT or img.name.startswith('.'):
                continue   # 숨김·AppleDouble(._*) 은 이미지가 아니다
            if seq_of(img.stem) in exclude_seqs:
                held += 1
                continue   # 홀드아웃 예약 시퀀스 — 학습에 절대 넣지 않는다
            if fraction < 1.0 and _hash1000('frac' + img.stem) >= fraction * 1000:
                skipped += 1
                continue
            lbl = src / 'labels' / sp / (img.stem + '.txt')
            if not lbl.exists():
                continue
            lines = [l for l in lbl.read_text().splitlines() if l.strip()]
            if not lines:
                continue
            for l in lines:
                counts[UNIFIED[int(l.split()[0])]] += 1
            _place(img, lines, dst_root, sp, img.stem, link)   # 이미 e63_ 접두사가 붙어 있음
            kept += 1
    return kept, skipped, held, counts


RESULTS = DRIVE / 'results' / RUN_NAME     # ★ 평가 결과를 공유 Drive 에 남긴다


def save_result(name, df, note=''):
    """평가표를 CSV 로 Drive 에 남긴다 — 화면 캡처에 의존하지 않는다.

    런타임이 회수되면 표는 사라지고, fair_eval 재실행은 데이터셋 재구축(약 30분)이
    필요하다(4차에서 실제로 겪었다). 재구축은 컴퓨팅 단위를 또 쓰므로,
    파일로 남기는 것이 유일하게 확실한 전달 방법이다.
    """
    RESULTS.mkdir(parents=True, exist_ok=True)
    p = RESULTS / f'{name}.csv'
    df.to_csv(p, index=False, encoding='utf-8-sig')
    if note:
        (RESULTS / f'{name}.txt').write_text(note.strip() + '\n')
    print(f'  → 저장 {p.name}  ({RESULTS})')
    return df


print('리매핑 유틸 로드 완료 · 평가 결과 저장 위치:', RESULTS)

In [ ]:
SOURCES = []
if USE_SH17:
    y = find_yaml('/content/raw', exclude=('css', 'hhw'))
    print('SH17 클래스 정의:', y or '(동봉 yaml 없음 → 공식 하드코딩 사용)')
    SOURCES.append(dict(name='sh17', root='/content/raw', yaml=str(y) if y else None,
                        names=SH17_NAMES, mapping=SH17_MAP, fraction=SH17_FRACTION))
if USE_CSS:
    SOURCES.append(dict(name='css', root='/content/raw/css', yaml='/content/raw/css/data.yaml',
                        names=None, mapping=CSS_MAP, fraction=CSS_FRACTION))
if USE_HHW:
    SOURCES.append(dict(name='hhw', root='/content/raw/hhw', yaml='/content/raw/hhw/data.yaml',
                        names=None, mapping=HHW_MAP, fraction=HHW_FRACTION))

total = Counter()
for s in SOURCES:
    names = load_names(s['yaml']) if s['yaml'] else s['names']
    idx_map = build_index_map(names, s['mapping'])
    mapped = {names[i]: UNIFIED[v] for i, v in idx_map.items() if v is not None}
    print(f"\n[{s['name']}] 원본 {len(names)}클래스 → 유지 {len(mapped)}종  (비율 {s['fraction']})")
    print('   순서:', names)
    print('   매핑:', mapped)
    if len(mapped) < len(set(s['mapping'].values())):
        print(f"   ⚠️ 매핑 누락 {sorted(set(s['mapping'].values()) - set(mapped.values()))}")
    pairs, n_aug, n_frac = collect_pairs(s['root'], fraction=s['fraction'])
    print(f"   이미지 {len(pairs)}장  (증강사본 제외 {n_aug} · 서브샘플 제외 {n_frac})")
    kept, dropped, counts = remap(pairs, DATA, idx_map, s['name'],
                                  head_filter=(s['name'] == 'sh17' and SH17_HEAD_FILTER))
    print(f'   유지 {kept} / 제외 {dropped}')
    print('  ', dict(counts))
    total.update(counts)

# ★ E63 — 이미 YOLO 형식이라 변환 없이 연결만
if USE_E63:
    # Drive 는 파일이 3만 개면 동기화가 매우 느리다 → 단일 tar 로 올려두고 여기서 푼다(약 1분)
    if not (E63_SRC / 'data.yaml').exists():
        assert E63_TAR.exists(), f'E63 아카이브 없음: {E63_TAR} — Drive 동기화 확인'
        E63_SRC.mkdir(parents=True, exist_ok=True)
        print(f'E63 아카이브 해제 중 ({E63_TAR.stat().st_size/1e9:.2f} GB)…')
        # macOS tar 가 넣은 애플 확장속성(com.apple.provenance)을 리눅스 tar 가 모른다.
        # 무해하지만 파일당 한 줄씩 3만 줄이 쏟아지므로 그 경고만 걸러낸다.
        !tar -xf "{E63_TAR}" -C "{E63_SRC}" 2>&1 | grep -v 'LIBARCHIVE.xattr' | head -20
        # 맥에서 만든 아카이브라 파일마다 AppleDouble(._*) 메타데이터가 딸려 있다.
        # 맥 tar 는 목록에서 감추지만 리눅스에서는 실제 파일로 풀리고, 확장자가
        # .jpg/.txt 라 라벨로 오인돼 UnicodeDecodeError 를 낸다.
        !find "{E63_SRC}" -name '._*' -delete
        !find "{E63_SRC}" -name '.DS_Store' -delete
        print('  해제 완료')
    assert (E63_SRC / 'data.yaml').exists(), f'E63 해제 실패: {E63_SRC}'
    # ★ 홀드아웃 시퀀스를 먼저 예약한다 — 학습에 한 프레임도 넣지 않는다
    ALL_SEQ, HOLD_SEQ = e63_reserve_seqs(E63_SRC, E63_HOLDOUT_SEQ_RATIO)
    print(f'\n시퀀스 {len(ALL_SEQ)}개 중 {len(HOLD_SEQ)}개를 홀드아웃으로 예약')
    assert 3 <= len(HOLD_SEQ) < len(ALL_SEQ), 'E63_HOLDOUT_SEQ_RATIO 를 조정할 것'
    # tar 를 /content 에 풀었으므로 이미 로컬 디스크다 → 링크로 충분(복사 불필요)
    kept, skipped, held, counts = link_yolo(E63_SRC, DATA, fraction=E63_FRACTION,
                                            link=True, exclude_seqs=HOLD_SEQ)
    print(f"[e63] 국내 AI Hub (비율 {E63_FRACTION})")
    print(f'   유지 {kept} / 서브샘플 제외 {skipped} / 홀드아웃 예약 {held}')
    print('  ', dict(counts))
    total.update(counts)

# ★ 7차-A · GDUT — E63 과 같은 구조(이미 통합 8클래스 YOLO 형식)라 연결만 한다.
#   tar 는 루트에 바로 images/ labels/ 가 있다(맥에서 tar -tf 로 확인 · 폴더 한 겹 없음 · ._ 0건).
#   val/ 643 은 tar 안에서 이미 해시 홀드아웃이고 GT 만 있다 → 별도 예약 없이 그대로 val 로 간다.
if USE_GDUT:
    if not (GDUT_SRC / 'data.yaml').exists():
        assert GDUT_TAR.exists(), f'GDUT 아카이브 없음: {GDUT_TAR} — Drive 동기화 확인'
        GDUT_SRC.mkdir(parents=True, exist_ok=True)
        print(f'GDUT 아카이브 해제 중 ({GDUT_TAR.stat().st_size/1e6:.0f} MB)…')
        !tar -xf "{GDUT_TAR}" -C "{GDUT_SRC}" 2>&1 | grep -v 'LIBARCHIVE.xattr' | head -20
        !find "{GDUT_SRC}" -name '._*' -delete
        !find "{GDUT_SRC}" -name '.DS_Store' -delete
        print('  해제 완료')
    assert (GDUT_SRC / 'data.yaml').exists(), (
        f'GDUT 해제 실패: {GDUT_SRC}\n'
        f'   내용: {sorted(p.name for p in GDUT_SRC.iterdir())[:8] if GDUT_SRC.is_dir() else "폴더 없음"}')
    # ★ 클래스 순서가 UNIFIED 와 같은지 눈으로가 아니라 코드로 확인한다(SH17 1차 사고)
    _gn = load_names(GDUT_SRC / 'data.yaml')
    assert _gn == UNIFIED, f'GDUT data.yaml 클래스 순서가 UNIFIED 와 다르다:\n  {_gn}\n  {UNIFIED}'
    # ★ 의사라벨이 val 에 새지 않았는지 — manifest 가 말하는 것을 라벨 파일로 다시 센다
    _vc = Counter()
    for _f in (GDUT_SRC / 'labels' / 'val').glob('*.txt'):
        for _l in _f.read_text().splitlines():
            if _l.strip():
                _vc[UNIFIED[int(_l.split()[0])]] += 1
    assert set(_vc) <= {'helmet', 'head'}, f'GDUT val 에 GT 밖 클래스가 있다(의사라벨 누수?): {dict(_vc)}'
    kept, skipped, held, counts = link_yolo(GDUT_SRC, DATA, fraction=GDUT_FRACTION, link=True)
    print(f"[gdut] GDUT-HWD GT + v5 의사라벨 (비율 {GDUT_FRACTION})")
    print(f'   유지 {kept} / 서브샘플 제외 {skipped}')
    print('  ', dict(counts))
    assert counts.get('helmet', 0) > 10000 and counts.get('person', 0) > 9000, (
        f'GDUT 라벨 수가 manifest(helmet 11,274 · person 10,055)와 동떨어졌다: {dict(counts)}')
    total.update(counts)

print('\n=== 통합 데이터셋 클래스별 인스턴스 ===')
for n in UNIFIED:
    c = total.get(n, 0)
    flag = '  ⚠️ 부족' if 0 < c < 500 else ('  ⚠️ 없음' if c == 0 else '')
    print(f'  {n:<12} {c:>7,}{flag}')

if total.get('head') and total.get('helmet'):
    print(f"\n★ helmet : head = 1 : {total['head']/total['helmet']:.2f}")
    print('   3차 1:0.75 · 4차 1:0.69 · 5차 1:0.91 → 7차-A 는 GDUT helmet +11,274 로 비율이 내려간다')
    print('   ⚠️ 1:1 을 크게 넘으면(head 과다) HHW 연결 실패를 의심할 것')

In [ ]:
splits = {p.name: len(list(p.glob('*'))) for p in (DATA / 'images').iterdir() if p.is_dir()}
print('split 별 이미지 수:', splits)

cfg = {'path': str(DATA), 'train': 'images/train', 'val': 'images/val',
       'names': {i: n for i, n in enumerate(UNIFIED)}}
if 'test' in splits:
    cfg['test'] = 'images/test'
DATA_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print()
print(DATA_YAML.read_text())

# ★ GPU 별 실측 기준으로 환산한다. 이전에는 1차 T4 상수만 써서 A100 에서도
#   42분/에포크(실제 8분대)로 찍혀 "20.9시간·15시간 초과" 오경보를 냈다.
#   ★ 09-14 정정 — T4 상수가 1차 값(11,521장·29.6분)이라 이 자료에서 48분/에포크(=24시간)를
#     찍었다. 09-13 실측은 20.4분이다. 두 배 이상 틀린 값으로 「완주 가능/불가」를 판정하고
#     있었다. 옛 상수는 imgsz·모델이 다른 실행의 것이라 이 노트북에 쓸 수 없다.
_REF = {'A100': (17_200, 8.7),      # 4차 실측
        'H100': (17_200, 6.0),      # 추정(A100 대비)
        'L4':   (11_521, 18.0),     # 추정(T4 대비 · 미검증)
        'T4':   (18_816, 20.4),     # ★ 09-13 실측 (results.csv time 열 · batch 16 · imgsz 640)
        }.get(next((k for k in ('A100', 'H100', 'L4', 'T4') if k in _gpu), ''),
              (18_816, 20.4))       # 기본 = T4 실측 (가장 흔한 배정)
est = splits.get('train', 0) / _REF[0] * _REF[1]
_hours = est * EPOCHS / 60
print(f'\n예상 소요 — 에포크당 약 {est:.1f}분 · {EPOCHS} 에포크 총 {_hours:.1f}시간  ({_gpu})')
if _hours > 15:
    print('⚠️ 15시간 초과 — 출처별 FRACTION 을 낮추거나 EPOCHS 를 줄일 것')

In [ ]:
# ★ 예약된 시퀀스로 홀드아웃을 만든다 — 학습에 단 한 프레임도 쓰이지 않았다
from collections import defaultdict

E63_HOLD = Path('/content/holdout_e63_seq')
HOLD_PER_SEQ = 200         # 시퀀스당 최대 장수. 인접 프레임은 거의 같지만
                           # 인스턴스 수가 적으면 신뢰구간이 넓어져 판정을 못 한다

if USE_E63:
    if E63_HOLD.exists():
        shutil.rmtree(E63_HOLD)
    (E63_HOLD / 'images' / 'val').mkdir(parents=True)
    (E63_HOLD / 'labels' / 'val').mkdir(parents=True)

    by_seq = defaultdict(list)
    for sp in ('train', 'val'):
        d = E63_SRC / 'images' / sp
        if not d.is_dir():
            continue
        for p in sorted(d.iterdir()):
            if p.suffix.lower() in IMG_EXT and not p.name.startswith('.') \
                    and seq_of(p.stem) in HOLD_SEQ:
                by_seq[seq_of(p.stem)].append(p)

    n = 0
    for s_, ps in sorted(by_seq.items()):
        ps = sorted(ps)
        for p in ps[::max(1, len(ps) // HOLD_PER_SEQ)][:HOLD_PER_SEQ]:
            lbl = p.parents[2] / 'labels' / p.parts[-2] / (p.stem + '.txt')
            if not lbl.exists():
                continue
            (E63_HOLD / 'images' / 'val' / p.name).symlink_to(p.resolve())
            shutil.copy2(lbl, E63_HOLD / 'labels' / 'val' / lbl.name)
            n += 1

    (E63_HOLD / 'data.yaml').write_text(yaml.safe_dump(
        {'path': str(E63_HOLD), 'train': 'images/val', 'val': 'images/val',
         'names': {i: x for i, x in enumerate(UNIFIED)}},
        allow_unicode=True, sort_keys=False))

    # ★ 누수 검사 — 홀드아웃이 학습셋과 한 장도 겹치면 안 된다
    tr = {p.stem for sp in ('train', 'val')
          for p in (DATA / 'images' / sp).glob('e63_*')}
    ho = {p.stem for p in (E63_HOLD / 'images' / 'val').iterdir()}
    assert not (tr & ho), f'홀드아웃 누수 {len(tr & ho)}장 — 예약 로직 확인'

    cnt = Counter()
    for f in (E63_HOLD / 'labels' / 'val').glob('*.txt'):
        for l in f.read_text().splitlines():
            if l.strip():
                cnt[UNIFIED[int(l.split()[0])]] += 1
    print(f'홀드아웃 시퀀스 {len(by_seq)}개 → {n:,}장 (학습 미노출)')
    print('  클래스:', dict(cnt))
    print('  누수 검사: 통과 (학습셋과 교집합 0)')


In [ ]:
# ── ★★ E63·HHW 에 person 자동 라벨 (도메인 얽힘 해소의 나머지 절반) ─────────
# ★ 7차-A — **gdut_ 는 대상이 아니다.** GDUT train 에는 v5 의사라벨 person 이 이미 들어 있다
#   (manifest: person 10,055). 여기서 또 붙이면 같은 사람에 박스가 둘 생긴다. 접두로 제외한다.
# E63 에는 person 라벨이 아예 없다. 그 결과 4차 모델은 "E63 처럼 보이는 장면"에서
# person 을 찾지 않았다(07-30 검증: E63 실이미지 head 1 · person 0). person 이 안 잡히면
# judge_ppe(person ↔ helmet 포함관계)로 작업자별 판정을 할 수 없다.
# COCO 사전학습 모델로 person 박스를 만들어 넣어 그 장면에서도 person 을 배우게 한다.
#
# ⚠️ train 분할에만 넣는다. val·홀드아웃에 넣으면 "자동 라벨과의 일치도"를 재는 순환이 된다.
#    이미 person 라벨이 있는 이미지(HHW 일부)는 건너뛴다 — 중복 박스를 만들지 않는다.
_MARK = DATA / '.person_pseudo_done'

# ★ 09-13 — 표시 파일만 믿지 않는다. 의사라벨은 라벨 .txt 에 **덧붙이는** 방식이라
#   리매핑 셀이 다시 돌면 붙인 person 줄이 지워지는데 **표시 파일은 별도 파일이라 살아남는다.**
#   그러면 「이미 했다」로 건너뛰고 person 이 절반인 채 학습이 시작된다(09-13 브라우저 실행에서 실제 발생 —
#   20,740 vs 목표 37,915. 라벨 수 게이트가 학습 전에 잡았다).
#   ⟹ 표시가 있어도 **실제 person 라벨 수를 세어** 모자라면 표시를 버리고 다시 붙인다.
if _MARK.exists():
    _n_now = sum(1 for _f in (DATA / 'labels' / 'train').glob('*.txt')
                 for _l in _f.read_text().splitlines()
                 if _l.strip() and int(_l.split()[0]) == UNIFIED_IDX['person'])
    _n_want = int((_MARK.read_text() or '0').strip())
    if _n_now < _n_want:
        print(f'⚠ 표시는 {_n_want:,}개인데 실제 person 은 {_n_now:,}개 — 리매핑이 덮어썼다. 표시를 버리고 다시 붙인다')
        _MARK.unlink()

if E63_PERSON_PSEUDO and not _MARK.exists():
    from ultralytics import YOLO as _COCO

    # ★ yolo11x(109MB) — 처음엔 yolo11n(5.4MB)을 썼는데 잘못된 트레이드오프였다.
    #   1회 오프라인 라벨링이고 A100 이 있으므로 속도가 아니라 정확도를 골라야 한다.
    #   실측(E63·HHW 균등 샘플 12장 · conf 0.5) n=15명 → x=19명(+27%).
    #   의사라벨의 오검출은 '유령 작업자' 를 학습시켜 judge_ppe 판정을 망치므로
    #   임계는 0.5 로 두고 recall 은 모델 성능으로 올린다.
    _coco = _COCO('yolo11x.pt')                    # COCO 80클래스 · person = 0
    P_IDX = UNIFIED_IDX['person']
    _lbl_dir = DATA / 'labels' / 'train'
    tgt = []
    for p in sorted((DATA / 'images' / 'train').iterdir()):
        if p.suffix.lower() not in IMG_EXT or p.name.split('_')[0] not in ('e63', 'hhw'):   # ★ gdut 제외
            continue
        lb = _lbl_dir / (p.stem + '.txt')
        if not lb.exists():
            continue
        if any(l.split()[0] == str(P_IDX) for l in lb.read_text().splitlines() if l.strip()):
            continue                               # 이미 person 라벨이 있다 → 건드리지 않는다
        tgt.append(p)
    print(f'person 자동 라벨 대상 {len(tgt):,}장 (train 분할·person 라벨 없는 것만 · conf {PSEUDO_CONF})')

    added = touched = 0
    B = 32                                     # x 모델이라 배치를 줄인다
    for i in range(0, len(tgt), B):
        _batch = tgt[i:i + B]
        # ★ 첫 구현이 11,589장에서 0개를 반환했다. 검출 자체는 정상임을 확인했으므로
        #   (E63·HHW 균등 샘플 12장 conf 0.5 에서 15~19명) 원인은 아래 둘이었다.
        #   ① 라벨 경로를 res.path 에서 유도하지 않는다 — 이미지가 심볼릭 링크라
        #      ultralytics 가 입력 경로를 그대로 돌려준다는 보장이 없다
        #      (HHW 는 링크명과 원본 stem 이 아예 다르다).
        #      입력 리스트와 zip 으로 1:1 대응시키면 그 의존이 사라진다.
        #   ② stream=True 를 쓰지 않는다 — 배치 추론엔 이득이 없고 변수만 늘린다.
        #   ⚠️ 진단 프로브는 tgt[:N] 이 아니라 전체 균등 분포로 뽑을 것 —
        #      영상 데이터는 앞쪽 N장이 같은 시퀀스의 연속 프레임이라 대표성이 없다.
        _res = _coco.predict([str(x) for x in _batch], imgsz=640, conf=PSEUDO_CONF,
                             classes=[0], verbose=False)
        for src, res in zip(_batch, _res):
            lb = _lbl_dir / (src.stem + '.txt')
            if not lb.exists():
                continue
            h, w = res.orig_shape
            new = []
            for x1, y1, x2, y2 in res.boxes.xyxy.tolist():
                bw, bh = (x2 - x1) / w, (y2 - y1) / h
                if bw <= 0 or bh <= 0:
                    continue
                new.append(f'{P_IDX} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} '
                           f'{bw:.6f} {bh:.6f}')
            if new:
                with open(lb, 'a') as f:
                    f.write('\n'.join(new) + '\n')
                added += len(new)
                touched += 1
        if (i // B) % 20 == 0:
            print(f'  {min(i + B, len(tgt)):,}/{len(tgt):,}')
    if added == 0:
        raise RuntimeError(
            'person 자동 라벨이 0개다 — 이대로 학습하면 5차의 절반(도메인 얽힘 해소)이 빠진다.\n'
            '   COCO 모델이 정말 아무것도 못 찾았는지 샘플 몇 장으로 직접 확인할 것.')
    _MARK.write_text(str(added))
    print(f'★ person 라벨 {added:,}개 추가 · 이미지 {touched:,}장 '
          f'(장당 {added / max(touched, 1):.1f}개)')
elif E63_PERSON_PSEUDO:
    print(f'person 자동 라벨 이미 적용됨 — 건너뜀 ({_MARK.read_text()}개)')
else:
    print('E63_PERSON_PSEUDO = False — 건너뜀')

# ── 최종 클래스 분포 재집계 (SH17 필터 + 자동 라벨 반영) ──────────────────
_final = Counter()
for _f in (DATA / 'labels').rglob('*.txt'):
    for _l in _f.read_text().splitlines():
        if _l.strip():
            _final[UNIFIED[int(_l.split()[0])]] += 1

print('\n=== 7차-A 최종 클래스별 인스턴스 (train+val · 자동 라벨 포함) ===')
for _n in UNIFIED:
    _c = _final.get(_n, 0)
    _flag = '  ⚠️ 부족' if 0 < _c < 500 else ('  ⚠️ 없음' if _c == 0 else '')
    print(f'  {_n:<12} {_c:>8,}{_flag}')
if _final.get('helmet'):
    print(f"\n★ helmet : head = 1 : {_final['head'] / _final['helmet']:.2f}   (5차 1:0.91)")


## 5. 학습

체크포인트는 Drive 에 저장된다. 세션이 끊기면 설정 셀에서 **`RESUME = True` 로만** 바꾸고
**설치 → 설정 → 데이터 → 리매핑 → 학습** 순서로 다시 실행한다(다른 값은 건드리지 말 것).

**학습 중 다른 셀을 실행하지 말 것** — 데이터 폴더를 건드리면 학습이 깨진다.

**협업 중이면 아래 잠금 셀을 먼저 실행한다** — 상대가 돌리는 중이면 거기서 멈춘다.

**화면의 ETA(`n:nn<n:nn`)는 무시할 것.** 순간 속도라 정체를 반영하지 못해 실제보다 빠르게 나온다.
완료된 에포크의 실제 소요로만 판단한다.

In [ ]:
# ── ★ 7차-A 라벨 수 검증 — 계획(train_v7.py PLAN['expect'] · 커밋 63a62b5b)과 실제가 다르면 학습하지 않는다
# 「507 이 없다」와 「507 이 있는데 안 셌다」가 같은 출력이 되지 않게 범위 밖 인덱스도 센다.
_EXPECT = {                                   # (하한, 상한) · None = 그쪽 경계 없음
    'safety_vest': (903 + 339, None),         # v5 903 + GDUT 의사라벨 339 (현장 35장 97 은 시험지라 뺐다)
    'helmet':      (12036 + 11274 - 800, 12036 + 11274 + 800),
    'head':        (12040 + 3256 - 800,  12040 + 3256 + 800),
    'person':      (29360 + 10055 - 1500, None),
}
_act, _oor = Counter(), Counter()
# ★ train 분할만 센다 — FIFTH(12,036 등)는 v5 runs/labels.jpg 에서 읽은 **train 라벨 수**다. val 까지 세면 GDUT val 3,248 등이
#   얹혀 상한을 넘긴다(09-13 첫 실행에서 실제로 걸림: helmet 30,190 · train 만 세면 23,310 = 정확히 12,036+11,274).
for _f in (DATA / 'labels' / 'train').glob('*.txt'):
    for _l in _f.read_text().splitlines():
        if _l.strip():
            _i = int(_l.split()[0])
            (_act if 0 <= _i < len(UNIFIED) else _oor)[UNIFIED[_i] if 0 <= _i < len(UNIFIED) else f'__범위밖_{_i}'] += 1
_bad = []
for _c, (_lo, _hi) in _EXPECT.items():
    _n = _act.get(_c, 0)
    if _lo is not None and _n < _lo: _bad.append(f'{_c} {_n:,} < 하한 {_lo:,}')
    if _hi is not None and _n > _hi: _bad.append(f'{_c} {_n:,} > 상한 {_hi:,}')
    print(f"  {_c:<12} {_n:>8,}   기대 [{_lo or '-':>7}, {_hi or '-':>7}]")
if _oor:
    _bad.append(f'범위 밖 인덱스 {dict(_oor)} — 507 이 섞였거나 9클래스 변환이 남았다')
assert not list((DATA / 'images' / 'train').glob('a507_*')), 'a507_ 파일이 학습셋에 있다 — USE_A507 이 False 인데 옛 /content/ppe3 가 남았다. DATA 를 지우고 다시'
# ★ 09-13 — 실험 조건 고정. kdhc(지역난방공사 100장·부분 순환 라벨)는 7차-B 재료다. 7차-A 에 섞이면 자료 효과가 안 갈린다.
for _pre in ('kdhc_',):
    assert not list((DATA / 'images').rglob(_pre + '*')), f'{_pre} 파일이 학습셋에 있다 — 7차-A 범위 밖 출처. DATA 를 지우고 다시'
assert not _bad, '[중단] 계획과 실제가 다르다 —\n  ' + '\n  '.join(_bad)
print('★ 라벨 수 검증 통과 · 507 부재 확인')


In [ ]:
# ── ★ 협업 잠금 — 같은 RUN_NAME 을 두 계정이 동시에 돌리면 양쪽 다 잃는다 ──────
# 학습 셀의 project= 가 공유 Drive 라 last.pt·best.pt·results.csv 를 같이 쓴다.
# exist_ok=True 여서 오류도 나지 않는다 — 몇 시간 뒤 조용히 망가진 것을 발견한다.
# 상대 세션이 살아있는지는 잠금 시각이 아니라 last.pt 갱신 시각으로 판정한다
# (에포크마다 갱신되므로 이것이 유일하게 신뢰할 수 있는 심장박동이다).
import time
from datetime import datetime, timedelta, timezone

assert RUNNER, 'RUNNER 를 설정 셀에 적을 것 — 비어 있으면 동시 실행을 막을 수 없다'
IDLE_MIN = 45          # ★ 09-14 상향. last.pt 는 에포크마다 갱신되는데 T4 에포크가 약 20분이라
                       #   30 이면 살아 있는 세션을 죽었다고 볼 여지가 있었다(에포크 1회 지연이면 걸린다).
                       #   45 면 T4 에서 두 에포크 가까이 멈춰야 죽은 것으로 본다.

KST = timezone(timedelta(hours=9))
_run_dir = RUNS / RUN_NAME
_run_dir.mkdir(parents=True, exist_ok=True)
_lock = _run_dir / 'OWNER.txt'
_last = _run_dir / 'weights' / 'last.pt'

if _lock.exists():
    _who, _, _when = _lock.read_text().strip().partition('|')
    if _last.exists():
        _idle = (time.time() - _last.stat().st_mtime) / 60
        print(f'직전 실행자 {_who} ({_when}) · last.pt 갱신 {_idle:.0f}분 전')
    else:
        _idle = (time.time() - _lock.stat().st_mtime) / 60
        print(f'직전 실행자 {_who} ({_when}) · last.pt 아직 없음 · 잠금 {_idle:.0f}분 전')
    if _who != RUNNER and _idle < IDLE_MIN:
        raise SystemExit(
            f'\n⛔ {_who} 의 세션이 살아있다(마지막 갱신 {_idle:.0f}분 전 · 기준 {IDLE_MIN}분).\n'
            f'   지금 같이 돌리면 서로의 last.pt 를 덮어써 양쪽 다 잃는다.\n'
            f'   → {_who} 에게 확인하고, 끝나거나 끊긴 뒤에 이어받을 것.')
    if _who != RUNNER:
        print(f'  {_who} 세션은 멈춰 있다 → 이어받는다')

_lock.write_text(f'{RUNNER}|{datetime.now(KST).isoformat(timespec="seconds")}\n')
print(f'잠금 기록 → {RUNNER}')

if not RESUME and _last.exists():
    print('\n⚠️ last.pt 가 있는데 RESUME=False 다 — 0에포크부터 다시 시작하며 덮어쓴다.')
    print('   이어받는 상황이면 설정 셀에서 RESUME=True 로 바꾸고 다시 올 것.')
elif RESUME and not _last.exists():
    print('\n⚠️ RESUME=True 인데 last.pt 가 없다 — 이어받을 체크포인트가 없다.')
    print('   첫 실행이면 RESUME=False 로 둘 것.')


In [ ]:
from ultralytics import YOLO

last = RUNS / RUN_NAME / 'weights' / 'last.pt'
_best_path = RUNS / RUN_NAME / 'weights' / 'best.pt'

# ★ 09-14 — 평가 전용 경로. 학습을 건너뛰고 아래 6~10장만 돌릴 때 쓴다.
if EVAL_ONLY:
    assert _best_path.exists(), f'best.pt 가 없다 — {_best_path}'
    _mb = _best_path.stat().st_size / 1e6
    print(f'평가 전용 모드 — 학습을 건너뛴다.\n  대상 {_best_path} ({_mb:.1f} MB)')
    model, results = YOLO(str(_best_path)), None
elif RESUME and last.exists():
    print('재개:', last)
    model = YOLO(str(last))
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        project=str(RUNS), name=RUN_NAME, exist_ok=True,
        patience=30, workers=WORKERS, seed=0, plots=True,
        scale=SCALE,   # ★ 5차: 스케일 내성. resume 시엔 체크포인트에 저장된 인자를 따른다
    )

best = RUNS / RUN_NAME / 'weights' / 'best.pt'
print('\nbest:', best)

## 6. 평가 — 1층 검출 지표 (회귀 감시 · 판정 아님)

**이 표로 7차-A 를 판정하지 않는다.** 6차는 여기서 P·R·mAP50·mAP50-95 네 지표가 전부 올랐는데
CAM-03 착용자 오경보가 +53% 였다. 판정 정본은 `train_v7.py` `GATE_COUNTS`(현장 35장·ppe.sample 박스 개수)이고
그 시험지는 Drive 에 없다 — [CCTV/ML] 이 독립 채점한다.

여기서 보는 것은 **회귀**뿐이다.

| 보는 것 | 기준 |
|---|---|
| `shoes`·`hands`·`gloves` 의 `R_변화` | **`-0.05` 아래면 부분 라벨 오염** — GDUT 에 이 셋은 의사라벨도 없다. SH17 이 받쳐야 산다 |
| `head` 의 `R_변화` | 내려가면 안 된다 — head 박스 = 위반 박스 |
| `person` | 자동 라벨이 GDUT 만큼 늘었으니 P 가 5차 `0.593` 근처면 정상 |

⚠️ **전체 val 지표는 쓰지 말 것.** gdut val 에는 `person`·`safety_vest` 라벨이 없어(GT 는 helmet·head 뿐)
정탐이 전부 오탐으로 계산된다. `fair_eval` 은 클래스마다 **그 클래스를 라벨링한 출처에서만** 잰다 — `SOURCE_CLASSES['gdut']` 가 그 경계다.


In [ ]:
import pandas as pd
from ultralytics import YOLO


def per_class(weights, data_yaml=None, imgsz=None, split='val'):
    m = YOLO(str(weights))
    r = m.val(data=str(data_yaml or DATA_YAML), imgsz=imgsz or IMGSZ, split=split, verbose=False)
    rows = [{'class': r.names[int(c)],
             'P': round(float(r.box.p[i]), 3), 'R': round(float(r.box.r[i]), 3),
             'mAP50': round(float(r.box.ap50[i]), 3), 'mAP50-95': round(float(r.box.ap[i]), 3)}
            for i, c in enumerate(r.box.ap_class_index)]
    return pd.DataFrame(rows).sort_values('R'), r


def _val_subset(prefixes, tag):
    """val 을 출처별로 잘라 임시 데이터셋을 만든다."""
    sub = Path(f'/content/val_{tag}')
    if sub.exists():
        shutil.rmtree(sub)
    (sub / 'images' / 'val').mkdir(parents=True)
    (sub / 'labels' / 'val').mkdir(parents=True)
    n = 0
    for p in sorted((DATA / 'images' / 'val').iterdir()):
        if p.name.split('_')[0] not in prefixes:
            continue
        lbl = DATA / 'labels' / 'val' / (p.stem + '.txt')
        if not lbl.exists():
            continue
        (sub / 'images' / 'val' / p.name).symlink_to(p.resolve())
        shutil.copy2(lbl, sub / 'labels' / 'val' / lbl.name)
        n += 1
    (sub / 'data.yaml').write_text(yaml.safe_dump(
        {'path': str(sub), 'train': 'images/val', 'val': 'images/val',
         'names': {i: x for i, x in enumerate(UNIFIED)}},
        allow_unicode=True, sort_keys=False))
    return sub, n


def fair_eval(weights):
    """클래스마다 그 클래스를 라벨링한 출처의 이미지에서만 평가한다.

    라벨이 없는 출처의 이미지에서는 올바른 검출도 오탐으로 계산된다.
    3차에서 person P 가 0.493 으로 찍혔으나 이 방식으로는 0.891 이었다.
    """
    groups = {}
    for cls in UNIFIED:
        pref = frozenset(s for s, cs in SOURCE_CLASSES.items() if cls in cs)
        if pref:
            groups.setdefault(pref, []).append(cls)
    out = []
    for pref, classes in groups.items():
        sub, n = _val_subset(pref, '_'.join(sorted(pref)))
        print(f'  {sorted(classes)}  ←  {sorted(pref)} val {n}장')
        df, _ = per_class(weights, data_yaml=sub / 'data.yaml')
        out.append(df[df['class'].isin(classes)])
    return pd.concat(out).sort_values('class').reset_index(drop=True)


print('=== 공정 평가 (클래스별로 라벨이 있는 출처에서만) ===')
df = fair_eval(best)

# 5차 공정 평가(fair_eval) 실측값 (P, R) — 같은 방식으로 낸 값이라 직접 비교된다.
# 출처: YOLO ML DATA/ppe_v5_eval_results/1_fair_eval.csv
V5 = {'person': (0.593, 0.844), 'head': (0.618, 0.804), 'helmet': (0.821, 0.749),
      'safety_vest': (0.713, 0.389), 'foot': (0.768, 0.079), 'shoes': (0.693, 0.451),
      'hands': (0.845, 0.749), 'gloves': (0.625, 0.410)}
df['P_5차'] = df['class'].map(lambda c: V5.get(c, (None, None))[0])
df['R_5차'] = df['class'].map(lambda c: V5.get(c, (None, None))[1])
df['R_변화'] = (df['R'] - df['R_5차']).round(3)
save_result('1_fair_eval', df)
display(df[['class', 'P', 'P_5차', 'R', 'R_5차', 'R_변화', 'mAP50', 'mAP50-95']])
print('\n⚠️ 이 표는 판정이 아니다. 6차는 여기서 네 지표가 전부 올랐는데 착용자 오경보가 +53% 였다.')
print('   7차-A 판정 = train_v7.py GATE_COUNTS(현장 35장·ppe.sample 박스 개수). 시험지는 Drive 에 없다.')
print('   여기서 볼 것은 회귀뿐 — R_변화 가 -0.05 아래인 클래스(특히 shoes·hands·gloves)가 있으면 멈춘다.')
print('   5차 홀드아웃 참고 — head P 0.768 · R 0.960 / helmet P 0.953 · R 0.575')


## 7. 평가 — 2층 착용 판정

`judge_ppe()` 는 person 박스를 요구하는데 **E63·HHW 에는 person 라벨이 없다.**
그래서 대부분의 데이터에서 그 방식으로는 정확도를 잴 수 없다.

대신 **이미지 단위**로 잰다. E63·CSS·HHW 는 `head`(미착용)와 `helmet`(착용)이 배타적이라
정답이 명확하다.

```
정답   라벨에 head 박스가 있다   → 미착용자 존재
예측   모델이 head 를 검출했다   → 미착용 알림 발생
```

**판정 게이트와 직접 대응한다** — `안전모 미착용 판정 recall ≥ 0.90 @ precision ≥ 0.85`.


In [ ]:
HEAD_I = UNIFIED_IDX['head']


def bare_head_eval(weights, data_root, confs=(0.15, 0.25, 0.35, 0.45, 0.55),
                   head_of=None, chunk=32, cls_i=None, gate=(0.90, 0.85)):
    """이미지 단위 '안전모 미착용' 알림의 정밀도·재현율을 임계별로 낸다.

    head_of  모델 클래스 dict → '맨머리로 칠 인덱스 집합'. 기본은 통합 인덱스(HEAD_I).
             현행 vyra 처럼 클래스 체계가 다른 모델을 같은 잣대로 잴 때 쓴다.
    chunk    한 번에 넣는 장수. 4차에서 2,166장 일괄 predict 가 CUDA 11.9GB 를
             할당하려다 죽었다 — 잘라 넣고 캐시를 비운다. 정답은 라벨 파일에서만 읽는다.

    ★ 정답 경로를 res.path 에서 유도하지 않는다. ultralytics 8.4.112 에서
      res.path 가 입력 경로와 어긋나 정답이 전부 False 로 잡혔다
      (인도메인 val 2,652장에서 "미착용자 있는 장면 0장" · 4차의 8.4.x 에서는 동작했다).
      입력 리스트와 zip 으로 1:1 대응시켜 그 의존을 없앤다.
      stream=True 도 제거한다 — 배치 추론엔 이득이 없고 변수만 늘린다.
    """
    import torch
    m = YOLO(str(weights))
    cls_i = HEAD_I if cls_i is None else cls_i
    head_ids = set(head_of(m.names)) if head_of else {cls_i}
    root = Path(data_root)
    imgs = [p for p in sorted((root / 'images' / 'val').iterdir())
            if p.suffix.lower() in IMG_EXT]
    gt, pred = [], []
    for i in range(0, len(imgs), chunk):
        _b = imgs[i:i + chunk]
        _r = m.predict([str(p) for p in _b], imgsz=IMGSZ, conf=min(confs), verbose=False)
        for src, res in zip(_b, _r):                    # ★ res.path 미사용
            lbl = root / 'labels' / 'val' / (src.stem + '.txt')
            gt.append(bool(lbl.exists()) and any(
                l.split() and int(l.split()[0]) == cls_i
                for l in lbl.read_text().splitlines() if l.strip()))
            pred.append([c for k, c in zip(res.boxes.cls.tolist(), res.boxes.conf.tolist())
                         if int(k) in head_ids])
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    rows = []
    for t in confs:
        hit = [any(c >= t for c in ps) for ps in pred]
        tp = sum(1 for g, h in zip(gt, hit) if g and h)
        fp = sum(1 for g, h in zip(gt, hit) if not g and h)
        fn = sum(1 for g, h in zip(gt, hit) if g and not h)
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        rows.append({'conf': t, 'P': round(P, 3), 'R': round(R, 3),
                     'F1': round(2 * P * R / (P + R), 3) if P + R else 0.0,
                     'TP': tp, 'FP': fp, 'FN': fn,
                     '게이트': '통과' if (R >= gate[0] and P >= gate[1]) else ''})
    assert sum(gt) > 0, (
        f'정답에 미착용자 있는 장면이 0장 ({len(gt):,}장 검사) — 라벨 경로가 어긋났다.\n'
        f'   {root}/labels/val 과 이미지 stem 을 대조할 것. 이 상태의 P·R 은 전부 무의미하다.')
    print(f'이미지 {len(gt):,}장 · [{UNIFIED[cls_i]}] 있는 장면 {sum(gt):,}장')
    return pd.DataFrame(rows)


print('=== 인도메인 val 기준 ===')
display(save_result('2_bare_head_indomain', bare_head_eval(best, DATA)))
print('★ 게이트: recall ≥ 0.90 @ precision ≥ 0.85')
print('   여기는 학습 분포다. 진짜 판정은 다음 장의 홀드아웃에서 한다.')


## 8. 홀드아웃 — E63 예약 시퀀스 (회귀 감시)

E63 홀드아웃은 **배포 도메인의 head·helmet 회귀 감시**다. 7차-A 의 성패 판정이 아니다 —
성패는 현장 35장·ppe.sample(시험 전용 · Drive 에 없음)에서 `train_v7.py` `GATE_COUNTS` 로 [CCTV/ML] 이 독립 채점한다.

여기서 볼 것 하나. **v5 홀드아웃 head P 0.768 · R 0.960 / helmet P 0.953 · R 0.575** 에서 helmet R 이 올라야 하고 head 가 무너지면 안 된다.


In [ ]:
assert E63_HOLD.exists(), '홀드아웃이 없다 — 4장의 홀드아웃 구성 셀을 먼저 실행할 것'

print('=== 1층 검출 (예약 시퀀스) ===')
df_ho, _ = per_class(best, data_yaml=E63_HOLD / 'data.yaml')
save_result('3_holdout_perclass', df_ho)
display(df_ho[df_ho['class'].isin(['head', 'helmet'])][['class', 'P', 'R', 'mAP50', 'mAP50-95']])

print('\n참고값 (같은 예약 홀드아웃)')
print('  4차                      head P 0.800 · R 0.893 · mAP50 0.906 / helmet P 0.946 · R 0.433')
print('  1차 교차 홀드아웃(HHW)   head P 0.181   ← 실패 판정의 근거\n')

print('=== 2층 미착용 알림 판정 (예약 시퀀스) ===')
display(save_result('4_holdout_bare_head', bare_head_eval(best, E63_HOLD)))
print('★ 게이트: recall ≥ 0.90 @ precision ≥ 0.85')


In [ ]:
# 임계 스윕 — 운영에 쓸 conf 를 고른다
import numpy as np

sweep = bare_head_eval(best, E63_HOLD, confs=tuple(np.round(np.arange(0.10, 0.71, 0.05), 2)))
display(sweep)

ok = sweep[(sweep['R'] >= 0.90) & (sweep['P'] >= 0.85)]
if len(ok):
    pick = ok.sort_values('F1', ascending=False).iloc[0]
    _note = (f"게이트 통과 · 통과 구간 {list(ok['conf'])} · 최고 F1 conf={pick['conf']} "
             f"(P {pick['P']} · R {pick['R']})")
    print(f"\n★ 게이트 통과 — 권장 conf = {pick['conf']}  (P {pick['P']} · R {pick['R']})")
    print('   4차는 0.10·0.15·0.20 통과 → 운영 conf 0.15 채택(새 현장 여유분)이었다')
else:
    best_f1 = sweep.sort_values('F1', ascending=False).iloc[0]
    _note = (f"게이트 미달 · 최고 F1 {best_f1['F1']} @ conf {best_f1['conf']} "
             f"(P {best_f1['P']} · R {best_f1['R']})")
    print(f"\n게이트 미달. 최고 F1 = {best_f1['F1']} @ conf {best_f1['conf']} "
          f"(P {best_f1['P']} · R {best_f1['R']})")
    print('   recall 이 부족하면 데이터·에포크, precision 이 부족하면 라벨 정합을 의심할 것')
save_result('5_holdout_sweep', sweep, note=_note)


## 9. 기준선 비교 — 현행 라이브 모델 (체크리스트 미결 해소)

판정 기준은 "얼마나 좋은가"가 아니라 **"현행보다 나은가"** 다.
현행 CAM-03 파이프라인의 원본 모델은 Hexmon `vyra_ppe.pt`(YOLOv8m 14클래스)다.

클래스 체계가 달라 1층 지표는 직접 비교가 안 된다. **같은 홀드아웃·같은 정답·같은 임계로
2층(이미지 단위 미착용 알림)을 잰다** — 교체 판단에 실제로 쓰는 잣대가 이것이다.


In [ ]:
!curl -sL -o /content/vyra_ppe.pt https://huggingface.co/Hexmon/vyra-yolo-ppe-detection/resolve/main/best.pt
baseline = YOLO('/content/vyra_ppe.pt')
print('현행 모델 클래스:', baseline.names)

# 현행 모델에서 '맨머리(미착용)' 에 해당하는 클래스만 골라 같은 잣대로 잰다
BASE_HEAD = {'nohardhat', 'nohelmet', 'head'}


def _base_head_ids(names):
    ids = {i for i, n in names.items() if _norm(n) in BASE_HEAD}
    assert ids, f'현행 모델에서 미착용 클래스를 못 찾음: {names}'
    return ids


print('\n=== 신규 (5차 yolo26) — 예약 홀드아웃 ===')
display(save_result('6_baseline_ours', bare_head_eval(best, E63_HOLD)))
print('\n=== 현행 (vyra_ppe) — 같은 홀드아웃 ===')
display(save_result('7_baseline_vyra',
                    bare_head_eval('/content/vyra_ppe.pt', E63_HOLD, head_of=_base_head_ids),
                    note=f'현행 모델 클래스: {baseline.names}'))
print('\n※ 같은 이미지·같은 정답·같은 임계다. 두 표의 P·R 을 그대로 나란히 읽으면 된다.')


## 10. 내보내기 → Jetson 이관

**코랩에서 TensorRT 엔진을 만들지 않는다.** 엔진은 기기·TensorRT 버전 종속이라 Jetson 에서 못 쓴다.

4차 실측 — 640 엔진 변환 490.9초 · 21.5MB · 유휴 **73.3 fps**. 절차는 검증돼 있다.

In [ ]:
out = DRIVE / 'weights'
out.mkdir(exist_ok=True)
dst = out / f'ppe_{RUN_NAME}.pt'
shutil.copy2(best, dst)
print('저장:', dst, f'({dst.stat().st_size/1e6:.1f} MB)')

# ★ 평가 결과 목록 — 다시 내려면 데이터셋 재구축에 컴퓨팅 단위를 또 쓴다. 여기 있으면 된다.
if RESULTS.exists():
    print(f'\n평가 결과 ({RESULTS})')
    for f in sorted(RESULTS.iterdir()):
        print(f'  {f.name}')
    print('  ⚠️ 이 목록이 비어 있으면 6~8장 평가 셀을 돌리지 않은 것이다.')
    print('     학습과 같은 세션에서 끝낼 것 — 나중에 하면 데이터셋 재구축 약 30분이 든다.')
if NC:
    print('\n⚠️ SH17(CC BY-NC-SA) · AI Hub(학습용 한정) 포함. 상용 빌드에 반입하지 말 것.')

print(f'''
다음 — Jetson Orin Nano Super 4GB (2026-07-27 정비 완료)

  # 이미 처리됨: 헤드리스 · 스왑 8GB · 디스크 49GB · ultralytics 8.4.107 · MAXN_SUPER
  # ★ 엔진 빌드 전 GUI 를 내릴 것 — 켜두면 TensorRT 가 최적 커널을 건너뛴다(실증됨)
  sudo systemctl isolate multi-user.target
  pkill -f orbbec_cctv_push.py; pkill -f gopro_ppe_push.py
  sudo jetson_clocks

  # ★★ 직사각으로 굽는다. 정사각(960x960)은 회색 여백 420px 이 착용자 오경보를 2.7배 낸다(09-12 확정).
  #    imgsz 는 형상과 똑같이 — 고정형상 엔진은 letterbox 가 그 형상으로 고정된다.
  python3 -c "from ultralytics import YOLO; \\
    YOLO('ppe_{RUN_NAME}.pt').export(format='engine', half=True, imgsz=(544, 960), workspace=2)"

  # 카메라 파이프라인을 켠 상태에서 정상상태 재측정 — 이게 진짜 숫자다
  jtop

  # 라이브 반영 — PPE_ENGINE 만 바꿔 릴레이를 재기동하면 CAM-03 에 즉시 반영된다
  # 배포 게이트(젯슨 · CAM-03 · head_conf 0.30) — 오경보 ≤ 6/125 AND 검출 ≥ 461/464. 통과 전엔 라이브에 올리지 않는다
  PPE_ENGINE=/home/jetson/ppe_{RUN_NAME}_rect544x960.engine nohup python3 phone_ppe_relay.py > relay.log 2>&1 &
''')